# House Prices: raw + QualityLivingArea blend 후보

## tl;dr

- Raw-year 예측 60.7%와 QualityLivingArea 예측 39.3%를 log1p 공간에서 혼합했다.
- Cross-fitted CV RMSLE는 0.144514 ± 0.043392다.
- Raw reference 대비 CV -0.005858, OOF -0.005075다.
- 개선 fold는 5/5이며 submission 형식 검사를 통과했다.
- Kaggle public/private 점수는 unverified다.

## Context & Methods

QualityLivingArea standalone 모델은 raw reference보다 CV가 나빴지만 OOF 잔차가
완전히 같지 않았다. 제출 슬롯을 보호하기 위해 raw 모델을 주축으로 유지하고
feature 모델을 한 개의 convex weight로만 섞는다.

### Key Assumptions

- blend는 log1p(SalePrice) 공간에서 수행한다.
- validation fold의 가중치는 나머지 네 fold OOF 행으로만 계산한다.
- 최종 test 가중치는 전체 OOF에서 계산한다.
- test target과 Kaggle 점수는 가중치 선택에 사용하지 않는다.
- 이 blend는 비교 submission이며 standalone PyTorch 최종 모델로 주장하지 않는다.

## Data

저장된 raw/feature OOF 예측과 두 submission을 사용한다. 입력 CSV의 Id 순서와
실제 train/test 행 순서를 다시 검증한다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import KFold

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
RAW_EXPERIMENT_ID = "BASEMLP-20260719-RAWYEAR-ONLY-NB-04"
FEATURE_EXPERIMENT_ID = "BASEMLP-20260719-FEAT-QUALAREA-NB-08"
CANDIDATE_EXPERIMENT_ID = "BASEMLP-20260719-BLEND-QUALAREA-NB-11-SUB-01"
FEATURE_NAME = "QualityLivingArea"

RAW_OOF_PATH = (
    ROOT / "reports" / f"basemlp_oof_{RAW_EXPERIMENT_ID}.csv"
)
FEATURE_OOF_PATH = (
    ROOT / "reports" / f"basemlp_oof_{FEATURE_EXPERIMENT_ID}.csv"
)
RAW_SUBMISSION_PATH = (
    ROOT / "submissions" /
    f"submission_{RAW_EXPERIMENT_ID}-SUB-01.csv"
)
FEATURE_SUBMISSION_PATH = (
    ROOT / "submissions" /
    f"submission_{FEATURE_EXPERIMENT_ID}-SUB-01.csv"
)
SUBMISSION_PATH = (
    ROOT / "submissions" /
    f"submission_{CANDIDATE_EXPERIMENT_ID}.csv"
)
NOTEBOOK_PATH = ROOT / "notebooks/feature_engineering/basemlp_blend_quality_living_interaction_candidate.ipynb"
SUMMARY_PATH = ROOT / "reports/basemlp_blend_quality_living_interaction_summary.json"
COMPARISON_PATH = ROOT / "reports/basemlp_blend_quality_living_interaction_comparison.json"

SEED = 42
N_SPLITS = 5


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(
        newline="", encoding="utf-8"
    ) as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {
            existing["experiment_id"] for existing in reader
        }
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        raise RuntimeError(
            f"Experiment ID already exists: {row['experiment_id']}"
        )
    with EXPERIMENTS_PATH.open(
        "a", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({
            field: row.get(field, "") for field in fieldnames
        })


raw_oof = pd.read_csv(RAW_OOF_PATH)
feature_oof = pd.read_csv(FEATURE_OOF_PATH)
raw_submission = pd.read_csv(RAW_SUBMISSION_PATH)
feature_submission = pd.read_csv(FEATURE_SUBMISSION_PATH)
train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

assert len(raw_oof) == len(feature_oof) == len(train) == 1460
assert raw_oof["Id"].equals(feature_oof["Id"])
assert raw_oof["Id"].equals(train["Id"])
assert np.array_equal(
    raw_oof["actual_log1p"].to_numpy(),
    feature_oof["actual_log1p"].to_numpy(),
)
assert len(raw_submission) == len(feature_submission) == len(test) == 1459
assert raw_submission["Id"].equals(feature_submission["Id"])
assert raw_submission["Id"].equals(test["Id"])
assert sample_submission["Id"].equals(test["Id"])


## Results

각 meta-validation fold에서 나머지 네 fold로 feature weight를 적합한다.

In [2]:
def fit_convex_weight(
    actual_log: np.ndarray,
    raw_log: np.ndarray,
    feature_log: np.ndarray,
) -> float:
    direction = feature_log - raw_log
    denominator = float(np.sum(direction ** 2))
    if denominator == 0:
        return 0.0
    numerator = -float(
        np.sum((raw_log - actual_log) * direction)
    )
    return float(np.clip(numerator / denominator, 0.0, 1.0))


actual_log = raw_oof["actual_log1p"].to_numpy(dtype=np.float64)
raw_oof_log = raw_oof["oof_log_prediction"].to_numpy(
    dtype=np.float64
)
feature_oof_log = feature_oof[
    "oof_log_prediction"
].to_numpy(dtype=np.float64)

folds = list(
    KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    ).split(raw_oof)
)
crossfit_blend_log = np.empty_like(actual_log)
crossfit_weights = []
fold_rows = []

for fold, (meta_train_index, meta_valid_index) in enumerate(
    folds, start=1
):
    weight = fit_convex_weight(
        actual_log[meta_train_index],
        raw_oof_log[meta_train_index],
        feature_oof_log[meta_train_index],
    )
    crossfit_weights.append(weight)
    prediction = (
        (1.0 - weight) * raw_oof_log[meta_valid_index]
        + weight * feature_oof_log[meta_valid_index]
    )
    crossfit_blend_log[meta_valid_index] = prediction
    fold_rows.append({
        "fold": fold,
        "feature_weight": weight,
        "raw_weight": 1.0 - weight,
        "rmsle": float(np.sqrt(np.mean(
            (
                actual_log[meta_valid_index]
                - prediction
            ) ** 2
        ))),
    })

final_feature_weight = fit_convex_weight(
    actual_log, raw_oof_log, feature_oof_log
)
final_raw_weight = 1.0 - final_feature_weight
fold_scores = np.asarray(
    [row["rmsle"] for row in fold_rows],
    dtype=np.float64,
)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
oof_rmsle = float(np.sqrt(np.mean(
    (actual_log - crossfit_blend_log) ** 2
)))
raw_fold_scores = []
for _, valid_index in folds:
    raw_fold_scores.append(float(np.sqrt(np.mean(
        (
            actual_log[valid_index]
            - raw_oof_log[valid_index]
        ) ** 2
    ))))
raw_cv_mean = float(np.mean(raw_fold_scores))
raw_oof_rmsle = float(np.sqrt(np.mean(
    (actual_log - raw_oof_log) ** 2
)))
improved_folds = int(np.sum(
    fold_scores < np.asarray(raw_fold_scores)
))

display(pd.DataFrame(fold_rows).round(6))
display(pd.Series({
    "feature": FEATURE_NAME,
    "final raw weight": final_raw_weight,
    "final feature weight": final_feature_weight,
    "cross-fitted CV mean": cv_mean,
    "cross-fitted CV std": cv_std,
    "cross-fitted OOF RMSLE": oof_rmsle,
    "raw CV mean": raw_cv_mean,
    "raw OOF RMSLE": raw_oof_rmsle,
    "improved folds": improved_folds,
}, name="blend validation"))


,fold,feature_weight,raw_weight,rmsle
0,1,0.396999,0.603001,0.137669
1,2,0.400275,0.599725,0.125185
2,3,0.424972,0.575028,0.220549
3,4,0.381902,0.618098,0.126168
4,5,0.363879,0.636121,0.112999


feature                   QualityLivingArea
final raw weight                   0.606608
final feature weight               0.393392
cross-fitted CV mean               0.144514
cross-fitted CV std                0.043392
cross-fitted OOF RMSLE             0.149635
raw CV mean                        0.150372
raw OOF RMSLE                       0.15471
improved folds                            5
Name: blend validation, dtype: object

## Submission

전체 OOF로 최종 weight를 계산해 raw와 feature test 로그 예측을 혼합한다.

In [3]:
raw_test_log = np.log1p(
    raw_submission["SalePrice"].to_numpy(dtype=np.float64)
)
feature_test_log = np.log1p(
    feature_submission["SalePrice"].to_numpy(dtype=np.float64)
)
blend_test_log = (
    final_raw_weight * raw_test_log
    + final_feature_weight * feature_test_log
)
blend_submission = sample_submission.copy()
blend_submission["SalePrice"] = np.expm1(blend_test_log)
blend_submission.to_csv(SUBMISSION_PATH, index=False)

checks = {
    "columns": blend_submission.columns.tolist() == [
        "Id", "SalePrice"
    ],
    "rows": len(blend_submission) == len(test),
    "id_order": blend_submission["Id"].equals(test["Id"]),
    "unique_ids": bool(blend_submission["Id"].is_unique),
    "finite_predictions": bool(np.isfinite(
        blend_submission["SalePrice"]
    ).all()),
    "positive_predictions": bool(
        blend_submission["SalePrice"].gt(0).all()
    ),
}
assert all(checks.values())

difference_vs_raw = (
    blend_submission["SalePrice"].to_numpy(dtype=np.float64)
    - raw_submission["SalePrice"].to_numpy(dtype=np.float64)
)
comparison = {
    "candidate_experiment_id": CANDIDATE_EXPERIMENT_ID,
    "raw_experiment_id": RAW_EXPERIMENT_ID,
    "feature_experiment_id": FEATURE_EXPERIMENT_ID,
    "feature_name": FEATURE_NAME,
    "blend_space": "log1p(SalePrice)",
    "weight_selection": (
        "closed-form log-MSE weight; cross-fitted for validation; "
        "all OOF rows for final test weight"
    ),
    "crossfit_weights": crossfit_weights,
    "final_raw_weight": final_raw_weight,
    "final_feature_weight": final_feature_weight,
    "fold_scores": fold_scores.tolist(),
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "oof_rmsle": oof_rmsle,
    "raw_cv_mean": raw_cv_mean,
    "raw_oof_rmsle": raw_oof_rmsle,
    "cv_delta_vs_raw": cv_mean - raw_cv_mean,
    "oof_delta_vs_raw": oof_rmsle - raw_oof_rmsle,
    "improved_folds": improved_folds,
    "submission_path": str(SUBMISSION_PATH.relative_to(ROOT)),
    "different_rows_vs_raw": int(np.count_nonzero(
        difference_vs_raw
    )),
    "mean_abs_difference_vs_raw": float(
        np.abs(difference_vs_raw).mean()
    ),
    "max_abs_difference_vs_raw": float(
        np.abs(difference_vs_raw).max()
    ),
    "submission_checks": checks,
    "kaggle_public_score": "unverified",
    "kaggle_private_score": "unverified",
}
COMPARISON_PATH.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
SUMMARY_PATH.write_text(
    json.dumps({
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "comparison": comparison,
        "source_paths": {
            "raw_oof": str(RAW_OOF_PATH.relative_to(ROOT)),
            "feature_oof": str(
                FEATURE_OOF_PATH.relative_to(ROOT)
            ),
            "raw_submission": str(
                RAW_SUBMISSION_PATH.relative_to(ROOT)
            ),
            "feature_submission": str(
                FEATURE_SUBMISSION_PATH.relative_to(ROOT)
            ),
        },
        "validation": {
            "notebook_executed_top_to_bottom": True,
            "external_script_dependency": False,
            "private_score": "unverified",
        },
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

append_experiment({
    "experiment_id": CANDIDATE_EXPERIMENT_ID,
    "datetime": datetime.now(timezone.utc).isoformat(),
    "objective": (
        f"Create a cross-fitted raw + {FEATURE_NAME} "
        "BaseMLP blend submission"
    ),
    "data_version": (
        f"train sha256={sha256(TRAIN_PATH)}; "
        f"test sha256={sha256(TEST_PATH)}"
    ),
    "split_strategy": (
        "Cross-fitted scalar blend over "
        "KFold(n_splits=5,shuffle=True,random_state=42) OOF"
    ),
    "folds": str(N_SPLITS),
    "seed": str(SEED),
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": (
        "Reuse validated raw and feature OOF/test predictions; "
        "fit scalar blend weight on meta-training OOF rows"
    ),
    "features": (
        f"Prediction blend: raw-year BaseMLP + "
        f"{FEATURE_NAME} BaseMLP"
    ),
    "model": "Log-space convex blend (comparison candidate)",
    "architecture": (
        f"{final_raw_weight:.6f}*raw BaseMLP + "
        f"{final_feature_weight:.6f}*{FEATURE_NAME} BaseMLP"
    ),
    "loss": "Closed-form MSE on OOF log1p predictions",
    "hyperparameters": json.dumps({
        "raw_experiment": RAW_EXPERIMENT_ID,
        "feature_experiment": FEATURE_EXPERIMENT_ID,
        "crossfit_weights": crossfit_weights,
        "final_raw_weight": final_raw_weight,
        "final_feature_weight": final_feature_weight,
        "blend_space": "log1p(SalePrice)",
        "external_script_dependency": False,
        "standalone_pytorch_final_model": False,
    }, ensure_ascii=False),
    "cv_mean": str(cv_mean),
    "cv_std": str(cv_std),
    "artifact_path": "; ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        str(SUBMISSION_PATH.relative_to(ROOT)),
        str(COMPARISON_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
    ]),
    "result": "submission_candidate_crossfit_blend",
    "interpretation": (
        f"Cross-fitted CV delta vs raw={cv_mean-raw_cv_mean:+.6f}; "
        f"OOF delta={oof_rmsle-raw_oof_rmsle:+.6f}; "
        f"improved folds={improved_folds}/5."
    ),
    "next_action": (
        "Record Kaggle score only after this exact CSV is submitted."
    ),
})

display(pd.Series(comparison, name="candidate summary"))


candidate_experiment_id            BASEMLP-20260719-BLEND-QUALAREA-NB-11-SUB-01
raw_experiment_id                           BASEMLP-20260719-RAWYEAR-ONLY-NB-04
feature_experiment_id                      BASEMLP-20260719-FEAT-QUALAREA-NB-08
feature_name                                                  QualityLivingArea
blend_space                                                    log1p(SalePrice)
weight_selection              closed-form log-MSE weight; cross-fitted for v...
crossfit_weights              [0.3969990439195679, 0.4002752375084412, 0.424...
final_raw_weight                                                       0.606608
final_feature_weight                                                   0.393392
fold_scores                   [0.13766926760107714, 0.12518549010839888, 0.2...
cv_mean                                                                0.144514
cv_std                                                                 0.043392
oof_rmsle                               

## Takeaways

- 최종 가중치는 raw 60.7%, QualityLivingArea 39.3%다.
- Validation weight는 해당 fold를 제외한 OOF 행에서 계산했다.
- CV 변화 -0.005858, OOF 변화 -0.005075, 개선 fold 5/5다.
- test target과 Kaggle 점수는 weight 선택에 사용하지 않았다.
- 이 파일은 blend 비교 후보이며 standalone PyTorch 최종 모델로 주장하지 않는다.
- 외부 프로젝트 스크립트 의존은 0건이다.